# Embeddings maken in Snowflake

Snowflake-versie van `02_upsert_embeddings.py`. In plaats van OpenAI gebruikt deze notebook **Snowflake Cortex**, zodat alles binnen Snowflake blijft (geen internettoegang nodig).

Draai deze notebook in Snowflake (database `FRUTA`, schema `PUBLIC`, warehouse `COMPUTE_WH`).

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Meertalig Cortex-model (geschikt voor Nederlands), 1024 dimensies
EMBEDDING_MODEL = "snowflake-arctic-embed-l-v2.0"
EMBEDDING_DIMENSION = 1024
# De naam van de kolom waarin de embeddings worden opgeslagen
EMBEDDING_COLUMN = "ARCTIC_EMBEDDING"

# (tabel, tekstkolom)
TABLES = [
    ("RECORDS", "NOTE"),
    # ("NOTES", "NOTE"),
]

In [ ]:
for table_name, text_column in TABLES:
    # Voeg de embedding-kolom toe als deze nog niet bestaat
    session.sql(
        f"ALTER TABLE {table_name} ADD COLUMN IF NOT EXISTS {EMBEDDING_COLUMN} VECTOR(FLOAT, {EMBEDDING_DIMENSION})"
    ).collect()

    # Maak embeddings voor rijen waar de embedding-kolom nog leeg is
    result = session.sql(
        f"""
        UPDATE {table_name}
        SET {EMBEDDING_COLUMN} = SNOWFLAKE.CORTEX.EMBED_TEXT_1024('{EMBEDDING_MODEL}', {text_column})
        WHERE {EMBEDDING_COLUMN} IS NULL
          AND {text_column} IS NOT NULL
          AND {text_column} != ''
        """
    ).collect()
    print(f"{table_name}: {result[0][0]} rijen bijgewerkt.")

In [ ]:
# Controle: aantal rijen met en zonder embedding per tabel
for table_name, _ in TABLES:
    print(
        session.sql(
            f"SELECT '{table_name}' AS tabel, COUNT({EMBEDDING_COLUMN}) AS met_embedding, COUNT(*) - COUNT({EMBEDDING_COLUMN}) AS zonder_embedding FROM {table_name}"
        ).collect()
    )